<a href="https://colab.research.google.com/github/keyvanmahmoudi/LLM_Engineering_Course/blob/main/solutions/Session15_dialogue_systems_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# پاسخ تمرین‌های جلسهٔ ۱۵: چت‌بات و سیستم دیالوگ وظیفه‌محور

این نوت‌بوک پاسخ پیشنهادی تمرین‌های انتهای [نوت‌بوک جلسه](../sessions/Session15_dialogue_systems.ipynb) است. قبل از دیدن پاسخ‌ها، خودتان تمرین‌ها را حل کنید؛ معمولاً بیش از یک راه‌حل درست وجود دارد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### آماده‌سازی

</div>

In [ ]:
!pip -q install openai
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/llm_utils.py
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/store_tools.py

from llm_utils import setup_llm, chat

client, MODEL = setup_llm("ollama")      # or: setup_llm("api")

In [ ]:
import json
from dataclasses import asdict, dataclass, field
from typing import Literal, Optional

from pydantic import BaseModel
from store_tools import get_product, place_order, search_products


class Update(BaseModel):
    """What the user's latest message says, in the context of the current state."""
    intent: Literal["order", "confirm", "deny", "other"]
    product_query: Optional[str]
    choice: Optional[int]
    quantity: Optional[int]
    color: Optional[str]
    city: Optional[str]


@dataclass
class State:
    product_query: Optional[str] = None
    product_id: Optional[str] = None
    candidates: list = field(default_factory=list)
    quantity: Optional[int] = None
    color: Optional[str] = None
    city: Optional[str] = None
    awaiting: Optional[str] = None          # "product", "choice", "color", "quantity", "city", "confirm"
    order: Optional[dict] = None


NLU_SYSTEM = """You update the state of an order dialogue for a Persian online store.
Given the current state, the assistant's last question and the user's new message, return what the NEW
message says. Leave a field null if the message does not mention it.
- intent: "order" when the user wants to buy or gives order details, "confirm" for yes/correct/ok
  (بله، آره، درسته، تأیید), "deny" for no/cancel (نه، خیر، لغو), otherwise "other".
- product_query: product words as written by the user (e.g. "گلکسی A55"), or null.
- choice: if the assistant listed numbered options and the user picks one ("دومی", "۲", "همون اولی"), its number.
- quantity: an integer if the user gives a number of items ("دو تا" -> 2, "یه" -> 1).
- color and city: in Persian."""


def understand(state, last_question, message):
    prompt = (f"STATE: {json.dumps(asdict(state), ensure_ascii=False)}\n"
              f"ASSISTANT ASKED: {last_question}\nUSER: {message}")
    response = client.chat.completions.create(
        model=MODEL, temperature=0,
        messages=[{"role": "system", "content": NLU_SYSTEM}, {"role": "user", "content": prompt}],
        response_format={"type": "json_schema",
                         "json_schema": {"name": "Update", "schema": Update.model_json_schema()}})
    return Update.model_validate_json(response.choices[0].message.content)


def track(state, update):
    """Dialogue state tracking: merge the update into the state."""
    if update.product_query and update.product_query != state.product_query:
        state.product_query, state.product_id, state.candidates = update.product_query, None, []
    if update.choice and state.candidates and 1 <= update.choice <= len(state.candidates):
        state.product_id = state.candidates[update.choice - 1]["product_id"]
    for slot in ("quantity", "color", "city"):
        if getattr(update, slot):
            setattr(state, slot, getattr(update, slot))
    return state


def policy(state, update):
    """Decide the next system action with explicit, testable rules."""
    if state.awaiting == "confirm":
        if update.intent == "confirm":
            try:
                state.order = place_order(state.product_id, state.quantity, state.city, state.color)
                return "placed", state.order
            except ValueError as e:
                return "order_failed", str(e)
        if update.intent == "deny":
            return "ask_change", None
    # 1. resolve the product (entity resolution with clarification)
    if state.product_id is None:
        if not state.product_query:
            return "ask_product", None
        state.candidates = search_products(state.product_query)
        in_stock = [c for c in state.candidates if c["stock"] > 0]
        if not in_stock:
            return "no_product", state.product_query
        if len(in_stock) == 1:
            state.product_id = in_stock[0]["product_id"]
        else:
            state.candidates = in_stock[:3]
            return "clarify_product", state.candidates
    product = get_product(state.product_id)
    # 2. validate and fill the remaining slots (colours compared without spaces / ZWNJ)
    same = lambda a, b: (a or "").replace("\u200c", "").replace(" ", "") == b.replace("\u200c", "").replace(" ", "")
    match = [c for c in product["colors"] if same(state.color, c)]
    if product["colors"] and not match:
        return "ask_color", (product, state.color)
    if match:
        state.color = match[0]                       # the catalogue spelling
    if not state.quantity:
        return "ask_quantity", None
    if state.quantity > product["stock"]:
        return "not_enough_stock", product
    if not state.city:
        return "ask_city", None
    return "confirm", product


def respond(action, data, state):
    """Template-based language generation: predictable, correct numbers."""
    if action == "ask_product":
        state.awaiting = "product"
        return "چه کالایی می‌خواهید سفارش بدهید؟"
    if action == "no_product":
        state.awaiting, state.product_query = "product", None
        return f"متأسفانه کالای موجودی برای «{data}» پیدا نکردم. لطفاً نام کالا را دقیق‌تر بنویسید."
    if action == "clarify_product":
        state.awaiting = "choice"
        options = "\n".join(f"{i}. {c['name']} ({c['price_toman']:,} تومان)" for i, c in enumerate(data, 1))
        return f"کدام کالا را می‌خواهید؟\n{options}"
    if action == "ask_color":
        product, wrong = data
        state.awaiting, state.color = "color", None
        prefix = f"رنگ «{wrong}» برای این کالا موجود نیست. " if wrong else ""
        return f"{prefix}{product['name']} در این رنگ‌ها موجود است: {'، '.join(product['colors'])}. کدام را می‌خواهید؟"
    if action == "ask_quantity":
        state.awaiting = "quantity"
        return "چند عدد می‌خواهید؟"
    if action == "not_enough_stock":
        state.awaiting, state.quantity = "quantity", None
        return f"از این کالا فقط {data['stock']} عدد موجود است. چند عدد می‌خواهید؟"
    if action == "ask_city":
        state.awaiting = "city"
        return "سفارش به کدام شهر ارسال شود؟"
    if action == "confirm":
        state.awaiting = "confirm"
        color = f"، رنگ {state.color}" if state.color else ""
        total = data["price_toman"] * state.quantity
        return (f"سفارش شما: {state.quantity} عدد {data['name']}{color}، ارسال به {state.city}. "
                f"مبلغ کالاها: {total:,} تومان. تأیید می‌کنید؟")
    if action == "ask_change":
        state.awaiting = None
        return "کدام مورد را می‌خواهید تغییر دهید؟"
    if action == "placed":
        state.awaiting = None
        return f"سفارش شما ثبت شد. کد پیگیری: {data['order_id']}. مبلغ: {data['total']:,} تومان."
    if action == "order_failed":
        state.awaiting = None
        return f"ثبت سفارش ممکن نشد: {data}"
    return "متوجه نشدم. لطفاً دوباره بگویید."


class OrderBot:
    def __init__(self):
        self.state, self.last = State(), "سلام! چه کالایی می‌خواهید سفارش بدهید؟"

    def turn(self, message):
        update = understand(self.state, self.last, message)
        track(self.state, update)
        action, data = policy(self.state, update)
        self.last = respond(action, data, self.state)
        return self.last, action

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۱

قصد جدیدی به نام `track_order` اضافه کنید: اگر کاربر کد پیگیری داد، وضعیت سفارش با ابزار `track_order` گزارش شود. NLU، سیاست و قالب پاسخ را به‌روز کنید.

</div>

In [ ]:
from store_tools import track_order


class TrackUpdate(BaseModel):
    intent: Literal["order", "track", "confirm", "deny", "other"]
    product_query: Optional[str]
    choice: Optional[int]
    quantity: Optional[int]
    color: Optional[str]
    city: Optional[str]
    order_id: Optional[str]


TRACK_NLU = NLU_SYSTEM.replace(
    'otherwise "other".',
    '"track" when the user asks about an existing order or gives a tracking code (NK + 6 digits), otherwise "other".'
) + "\n- order_id: a tracking code such as NK482913, or null."


class TrackingBot(OrderBot):
    def turn(self, message):
        prompt = (f"STATE: {json.dumps(asdict(self.state), ensure_ascii=False)}\n"
                  f"ASSISTANT ASKED: {self.last}\nUSER: {message}")
        response = client.chat.completions.create(
            model=MODEL, temperature=0,
            messages=[{"role": "system", "content": TRACK_NLU}, {"role": "user", "content": prompt}],
            response_format={"type": "json_schema",
                             "json_schema": {"name": "TrackUpdate", "schema": TrackUpdate.model_json_schema()}})
        u = TrackUpdate.model_validate_json(response.choices[0].message.content)
        if u.intent == "track":
            if not u.order_id:
                self.last = "لطفاً کد پیگیری سفارش را بنویسید (مثل NK482913)."
                return self.last, "ask_order_id"
            try:
                o = track_order(u.order_id)
                self.last = f"سفارش {o['order_id']}: وضعیت «{o['status']}»، ارسال به {o['city']}."
            except ValueError:
                self.last = f"سفارشی با کد {u.order_id} پیدا نکردم."
            return self.last, "tracked"
        update = Update(**{k: v for k, v in u.model_dump().items() if k in Update.model_fields}
                        | {"intent": u.intent if u.intent != "track" else "other"})
        track(self.state, update)
        action, data = policy(self.state, update)
        self.last = respond(action, data, self.state)
        return self.last, action


bot = TrackingBot()
for user in ["سفارشم کجاست؟", "NK519027", "یه شارژر انکر هم می‌خوام"]:
    reply, action = bot.turn(user)
    print(f"USER: {user}\nBOT [{action}]: {reply}\n")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

قصد جدید سه جزء را تغییر داد: NLU، که حالا `order_id` و قصد `track` را می‌شناسد، سیاست، که پیگیری را بدون دست زدن به وضعیت سفارش انجام می‌دهد، و پاسخ. پیگیری روی وضعیت سفارش در جریان اثری ندارد. اگر کاربر وسط ثبت سفارش، سفارش قبلی‌اش را پیگیری کند، گفتگوی ثبت سفارش از دست نمی‌رود. این یکی از مزیت‌های وضعیت صریح است.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۲

کاربر وسط گفتگو نظرش را عوض می‌کند: «نه، سه تا می‌خوام» یا «اصلاً آیفون می‌خوام». بررسی کنید سیستم فعلی این تغییرها را درست ردیابی می‌کند یا نه و اشکال‌ها را رفع کنید.

</div>

In [ ]:
def run(bot, messages):
    for user in messages:
        reply, action = bot.turn(user)
        print(f"USER: {user}\nBOT [{action}]: {reply}")
    print("state:", {k: v for k, v in asdict(bot.state).items() if k != "candidates"}, "\n")


run(OrderBot(), ["دو تا پاوربانک شیائومی مشکی برای رشت", "نه، سه تا می‌خوام", "بله"])
run(OrderBot(), ["یه گلکسی A55 آبی برای تهران", "اصلاً آیفون ۱۶ می‌خوام"])

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

در نسخهٔ جلسه، تغییر محصول درست کار می‌کند: `track` با دیدن `product_query` جدید، کالای قبلی را پاک می‌کند و رنگ قبلی هم دوباره با رنگ‌های کالای جدید سنجیده می‌شود. ولی تغییر تعداد در مرحلهٔ تأیید مشکل دارد: «نه، سه تا می‌خوام» قصد `deny` دارد. `track` تعداد را به‌روز می‌کند، ولی `policy` فقط «کدام مورد را تغییر دهید؟» را می‌پرسد و کاربر باید دوباره بگوید. اصلاح: اگر پیام رد، اطلاعات جدید هم دارد، مستقیماً دوباره خلاصه را برای تأیید نشان بده.

</div>

In [ ]:
_original_policy = policy


def policy(state, update):
    has_new_info = any([update.product_query, update.quantity, update.color, update.city, update.choice])
    if state.awaiting == "confirm" and update.intent == "deny" and has_new_info:
        state.awaiting = None                      # re-validate everything and confirm again
        update = update.model_copy(update={"intent": "order"})
    return _original_policy(state, update)


run(OrderBot(), ["دو تا پاوربانک شیائومی مشکی برای رشت", "نه، سه تا می‌خوام", "بله"])

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۳

تابع `respond` را با نسخه‌ای جایگزین کنید که پاسخ را با مدل زبانی تولید کند، با اقدام و داده به‌عنوان ورودی. پاسخ‌ها طبیعی‌تر می‌شوند؟ آیا عددها و نام‌ها همیشه درست می‌مانند؟

</div>

In [ ]:
NLG_SYSTEM = ("You write the reply of a polite Persian shopping assistant. You get the ACTION the system decided "
              "and its DATA. Write one or two natural sentences in Persian. Copy every number, price, product "
              "name and option EXACTLY from the data. Do not add any information that is not in the data.")


def respond_llm(action, data, state):
    template = respond(action, data, state)            # also updates state.awaiting
    return chat(client, MODEL, f"ACTION: {action}\nDATA (template reply): {template}", system=NLG_SYSTEM,
                temperature=0.3), template


import re

bot = OrderBot()
problems = 0
for user in ["یه گوشی سامسونگ می‌خوام", "اولی", "آبی", "دو تا", "مشهد"]:
    update = understand(bot.state, bot.last, user)
    track(bot.state, update)
    action, data = policy(bot.state, update)
    natural, template = respond_llm(action, data, bot.state)
    bot.last = natural
    numbers = lambda t: set(re.findall(r"[\d,٬۰-۹]{4,}", t))
    missing = numbers(template) - numbers(natural)
    problems += bool(missing)
    print(f"USER: {user}\n  template: {template}\n  llm     : {natural}\n  missing numbers: {missing or '-'}\n")
print("replies with missing or changed numbers:", problems)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

پاسخ‌های مدل معمولاً طبیعی‌ترند، ولی هر بازنویسی یک فرصت خطاست: عدد ممکن است گرد شود، ارقام فارسی و انگلیسی عوض شوند، یا گزینه‌ای از فهرست جا بیفتد. بررسی خودکار عددها، مثل کد بالا، حداقل محافظ است. راه میانه که در عمل رایج است: **قالب برای جمله‌های حاوی عدد و تأیید** (قیمت، خلاصهٔ سفارش، کد پیگیری)، و مدل برای جمله‌های آزاد مثل خوش‌آمدگویی، همدلی و پاسخ به پرسش‌های خارج از جریان.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۴

شبیه‌ساز را روی ۱۰ هدف، هر کدام ۳ بار، اجرا کنید و همهٔ شکست‌ها را دسته‌بندی کنید: خطای NLU، خطای سیاست یا خطای شبیه‌ساز. کدام جزء بیشترین بهبود را لازم دارد؟

</div>

In [ ]:
import pandas as pd

SIMULATOR = """You are a customer of an Iranian online store, chatting in informal Persian.
Your goal: buy {quantity} x {product}, colour {color}, delivered to {city}.
Rules: write short messages. Do NOT give all details at once; start with only the product. Answer the
assistant's questions according to your goal. If the assistant offers numbered options, pick the one that
matches your goal. When the summary matches your goal, confirm with «بله». Otherwise say what is wrong."""

GOALS = [
    {"product": "گلکسی A55", "product_id": "P1001", "quantity": 1, "color": "آبی", "city": "آمل"},
    {"product": "هدفون سونی", "product_id": "P5001", "quantity": 2, "color": "نقره‌ای", "city": "تهران"},
    {"product": "پاوربانک شیائومی", "product_id": "P5005", "quantity": 3, "color": "مشکی", "city": "رشت"},
    {"product": "ساعت هوشمند سامسونگ", "product_id": "P5003", "quantity": 1, "color": "سبز", "city": "یزد"},
    {"product": "آیفون 16", "product_id": "P1005", "quantity": 1, "color": "صورتی", "city": "مشهد"},
    {"product": "ردمی نوت 14 پرو", "product_id": "P1003", "quantity": 2, "color": "بنفش", "city": "قم"},
    {"product": "شارژر انکر", "product_id": "P5004", "quantity": 4, "color": "سفید", "city": "کرج"},
    {"product": "لپ‌تاپ لنوو", "product_id": "P2001", "quantity": 1, "color": "خاکستری", "city": "تبریز"},
    {"product": "لباسشویی سامسونگ", "product_id": "P4002", "quantity": 1, "color": "سفید", "city": "اصفهان"},
    {"product": "ردمی بادز 6", "product_id": "P5002", "quantity": 5, "color": "مشکی", "city": "ساری"},
]
squash = lambda x: (x or "").replace("\u200c", "").replace(" ", "")


def simulate(goal, max_turns=10):
    bot = OrderBot()
    history = [{"role": "system", "content": SIMULATOR.format(**goal)}, {"role": "user", "content": bot.last}]
    transcript = []
    for turn in range(1, max_turns + 1):
        user = chat(client, MODEL, messages=history, temperature=0.7)
        reply, action = bot.turn(user)
        transcript.append((user, action, reply))
        history += [{"role": "assistant", "content": user}, {"role": "user", "content": reply}]
        if action == "placed":
            s = bot.state
            wrong = [k for k, a, b in [("product", s.product_id, goal["product_id"]), ("quantity", s.quantity, goal["quantity"]),
                                       ("color", squash(s.color), squash(goal["color"])), ("city", s.city, goal["city"])] if a != b]
            return {"success": not wrong, "turns": turn, "wrong": wrong, "last_action": action, "transcript": transcript}
    return {"success": False, "turns": max_turns, "wrong": [], "last_action": transcript[-1][1], "transcript": transcript}


runs = [{"goal": g["product"], **simulate(g)} for g in GOALS for _ in range(3)]
results = pd.DataFrame(runs)
print(f"task success: {results.success.mean():.0%}   mean turns: {results.turns.mean():.1f}")
results[["goal", "success", "turns", "wrong", "last_action"]]

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

برای دسته‌بندی هر شکست، گفتگوی آن را بخوانید:

</div>

In [ ]:
for _, r in results[~results.success].head(5).iterrows():
    print(f"=== {r.goal} (wrong: {r.wrong}, last action: {r.last_action})")
    for user, action, reply in r.transcript:
        print(f"  customer: {user}\n  bot [{action}]: {reply[:120]}")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

راهنمای دسته‌بندی:

| نشانه | دسته |
|---|---|
| مشتری اطلاعات درست داده ولی وضعیت چیز دیگری ثبت کرده (`wrong` غیرخالی) | خطای NLU |
| سیستم سؤالی را تکرار می‌کند یا به تأیید نمی‌رسد با اینکه همهٔ اسلات‌ها پر است | خطای سیاست |
| مشتری اطلاعاتی برخلاف هدفش داده یا جواب بی‌ربط داده | خطای شبیه‌ساز |
| کالا با این نام پیدا نمی‌شود (`no_product` پشت سر هم) | خطای جستجوی کالا (تطبیق نام) |

در این سیستم، بیشتر خطاها معمولاً از **تطبیق نام کالا** و **NLU** می‌آیند: نام‌های محاوره‌ای مثل «بادز» که جستجوی کلیدواژه‌ای ما پیدا نمی‌کند، و رنگ یا تعدادی که در پاسخ‌های کوتاه گم می‌شود. جستجوی معنایی به‌جای کلیدواژه‌ای (جلسهٔ ۶) و چند مثال Few-shot در NLU (جلسهٔ ۴) دو بهبود طبیعی‌اند. اثر هر بهبود را با همین شبیه‌ساز بسنجید.

</div>